# 第25章 最新LLMの部品を移植する

RoPE・RMSNorm・SwiGLU・MoEを自分のGPTに入れて、公平に比べる

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 準備と、比べる相手（ベースライン）


In [ ]:
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/minigpt.py
import re, math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
from minigpt import GPT

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/master/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    text = urllib.request.urlopen(url).read().decode("cp932")
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]
T = 64

def get_batch(data, g):
    ix = torch.randint(0, len(data) - T, (32,), generator=g)
    return torch.stack([data[i:i + T] for i in ix]), torch.stack([data[i + 1:i + T + 1] for i in ix])

@torch.no_grad()
def val_loss(model):
    model.eval()
    starts = range(0, len(val_data) - T - 1, T)
    xs = torch.stack([val_data[i:i + T] for i in starts])
    ys = torch.stack([val_data[i + 1:i + T + 1] for i in starts])
    loss = model(xs, ys)[1].item()
    model.train()
    return loss

def experiment(make_model, seed, steps=1500):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = make_model()
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    for step in range(steps):
        x, y = get_batch(train_data, g)
        _, loss = model(x, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return val_loss(model), sum(p.numel() for p in model.parameters())

base = lambda: GPT(V, block_size=T, n_embd=64, n_head=4, n_layer=2, dropout=0.0)   # 現代版に合わせてドロップアウトなし
results = {"ベースライン（GPT-2型）": [experiment(base, seed) for seed in [0, 1]]}
for name, rs in results.items():
    print(name, "検証損失：", [round(r[0], 3) for r in rs], " パラメータ：", rs[0][1])

## 部品1：RMSNorm


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
print(RMSNorm(4)(x))

## 部品2：SwiGLU


In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False)   # 中身
        self.w3 = nn.Linear(dim, hidden, bias=False)   # 門番
        self.w2 = nn.Linear(hidden, dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

## 部品3：RoPE（回転位置埋め込み）


In [ ]:
def rope_cache(T, hs, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, hs, 2).float() / hs))   # ペアごとの回転の速さ
    angles = torch.outer(torch.arange(T).float(), freqs)            # [T, hs/2]：位置 × 速さ
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]            # 数を2つずつペアにする
    T = x.shape[-2]
    c, s = cos[:T], sin[:T]
    out = torch.stack([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1)   # 各ペアを回転
    return out.flatten(-2)

In [ ]:
hs = 16
cos, sin = rope_cache(200, hs)
torch.manual_seed(0)
q, k = torch.randn(hs), torch.randn(hs)

def score_at(m, n):
    qm = apply_rope(q.expand(200, hs), cos, sin)[m]   # 位置mに置いたQuery
    kn = apply_rope(k.expand(200, hs), cos, sin)[n]   # 位置nに置いたKey
    return (qm @ kn).item()

print(f"位置(5, 3)の点数：{score_at(5, 3):.4f}")
print(f"位置(105, 103)の点数：{score_at(105, 103):.4f}")
print(f"位置(5, 1)の点数：{score_at(5, 1):.4f}")

## 現代版ミニGPTを組み立てる


In [ ]:
class ModernAttention(nn.Module):
    def __init__(self, dim, n_head, block_size):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        cos, sin = rope_cache(block_size, dim // n_head)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v)]
        q, k = apply_rope(q, self.cos, self.sin), apply_rope(k, self.cos, self.sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class ModernBlock(nn.Module):
    def __init__(self, dim, n_head, block_size):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(dim), RMSNorm(dim)
        self.attn = ModernAttention(dim, n_head, block_size)
        self.ffn = SwiGLU(dim, hidden=int(dim * 8 / 3))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class ModernGPT(nn.Module):
    def __init__(self, V, block_size, dim=64, n_head=4, n_layer=2):
        super().__init__()
        self.tok_emb = nn.Embedding(V, dim)
        self.blocks = nn.Sequential(*[ModernBlock(dim, n_head, block_size) for _ in range(n_layer)])
        self.norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, V, bias=False)
        for p in self.parameters():
            if p.dim() == 2:
                nn.init.normal_(p, std=0.02)

    def forward(self, idx, targets=None):
        x = self.blocks(self.tok_emb(idx))
        logits = self.lm_head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

modern = lambda: ModernGPT(V, block_size=T)
results["現代版（RMSNorm＋SwiGLU＋RoPE）"] = [experiment(modern, seed) for seed in [0, 1]]
for name, rs in results.items():
    losses = [r[0] for r in rs]
    print(f"{name}：検証損失 {[round(l, 3) for l in losses]}  平均 {sum(losses) / len(losses):.3f}  パラメータ {rs[0][1]:,}")

## MoE：専門家を使い分ける


In [ ]:
class TinyMoE(nn.Module):
    def __init__(self, dim, n_experts=4, hidden=128):
        super().__init__()
        self.router = nn.Linear(dim, n_experts, bias=False)       # どの専門家に任せるか決める係
        self.experts = nn.ModuleList([SwiGLU(dim, hidden) for _ in range(n_experts)])

    def forward(self, x):
        B, T, C = x.shape
        flat = x.view(-1, C)
        gate = F.softmax(self.router(flat), dim=-1)                # 各トークンの、専門家ごとの重み
        weight, choice = gate.max(dim=-1)                          # 一番の専門家を1人だけ選ぶ（top-1）
        out = torch.zeros_like(flat)
        for e, expert in enumerate(self.experts):
            mask = choice == e
            if mask.any():
                out[mask] = weight[mask, None] * expert(flat[mask])
        self.last_choice = choice
        return out.view(B, T, C)

torch.manual_seed(0)
moe = TinyMoE(64)
x = torch.randn(2, 16, 64)
y = moe(x)
print(y.shape)
print("各専門家が担当したトークン数：", torch.bincount(moe.last_choice, minlength=4).tolist())

## 最新研究の地図


In [ ]:
rewards = torch.tensor([1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0])   # 8通りの答えの正誤
advantage = (rewards - rewards.mean()) / (rewards.std() + 1e-6)
print("グループの平均：", rewards.mean().item())
print("アドバンテージ：", [round(a, 2) for a in advantage.tolist()])